In [ ]:
pip install "torch<2.6"

In [ ]:
pip install neuralprophet

In [ ]:
from neuralprophet import NeuralProphet
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import glob
import os

import torch
import neuralprophet.configure as npc

torch.serialization.add_safe_globals([
    npc.ConfigSeasonality,
    npc.Season,
])

In [ ]:
df = pd.read_csv('../scr/data/cleaned_rat_sightings_data/daily_borough_rs.csv')
df['created_date'] = pd.to_datetime(df['created_date']) 
df = df[df['borough']=='MANHATTAN']
full_dates = pd.date_range('2020-01-01', '2025-12-31', freq='D')
full_index = pd.MultiIndex.from_product([['MANHATTAN'], full_dates], names=['borough', 'created_date'])
df = df.set_index(['borough', 'created_date']).reindex(full_index).fillna({'count': 0}).reset_index()
df['count'] = df['count'].astype(int)
df.drop(columns=['borough'], inplace=True)
df = df.rename(columns = {'created_date' : 'ds', 'count':'y'})


In [ ]:
m = NeuralProphet(
    learning_rate=0.01,   # any fixed value
    epochs=100
)
metrics = m.fit(df)



In [ ]:
predicted = m.predict(df)
forecast = m.predict(df)

In [ ]:
m.plot(forecast)

In [ ]:
m.plot_components(forecast)

In [ ]:
m.plot_parameters()

## 80/20 Train Test Split

In [ ]:
m = NeuralProphet(seasonality_mode="additive", yearly_seasonality=True, 
                  weekly_seasonality=True, 
                  loss_func = 'MSE',
                  learning_rate=0.1)

df_train, df_test = m.split_df(df=df, freq="D", valid_p=0.2)

metrics_train = m.fit(df=df_train, freq="D")
metrics_test = m.test(df=df_test)

metrics_test

In [ ]:
df_test

## Cross-Validation

In [ ]:
METRICS = ["MAE", "RMSE"]
METRICS_VAL = ["MAE_val", "RMSE_val"]
params = {"seasonality_mode": "additive", "learning_rate": 0.1}

T = len(df)

fold_pct = 14 / T
fold_overlap_pct = 1 - (30 / (730 + 14))

folds = NeuralProphet(**params).crossvalidation_split_df(
    df,
    freq="D",
    k=5,
    fold_pct=fold_pct,
    fold_overlap_pct=fold_overlap_pct
)

In [ ]:
metrics_train = pd.DataFrame(columns=METRICS)
metrics_test = pd.DataFrame(columns=METRICS_VAL)

for df_train, df_test in folds:
    m = NeuralProphet(**params)
    m.set_plotting_backend("plotly-static")
    train = m.fit(df=df_train, freq="D")
    test = m.test(df=df_test)
    
    metrics_train = pd.concat(
        [metrics_train, train[METRICS].iloc[[-1]]],
        ignore_index=True
    )

    metrics_test = pd.concat(
        [metrics_test, test[METRICS_VAL].iloc[[-1]]],
        ignore_index=True
    )

In [ ]:
metrics_test

In [ ]:
metrics_train

In [ ]:
import pandas as pd
from datetime import timedelta

rows = []

# Prophet-style CV parameters
initial = pd.Timedelta("730 days")
period = pd.Timedelta("30 days")
horizon = pd.Timedelta("14 days")

# Define cutoffs exactly like Prophet
cutoff = df["ds"].min() + initial
end = df["ds"].max()

while cutoff + horizon <= end:

    # Train / test split
    df_train = df[df["ds"] <= cutoff]
    df_test = df[(df["ds"] > cutoff) & (df["ds"] <= cutoff + horizon)]

    # Fit NeuralProphet
    m = NeuralProphet(**params)
    m.fit(df_train, freq="D")

    # Forecast on test window
    forecast = m.predict(df_test)

    # Merge truth + forecast
    df_eval = (
        df_test[["ds", "y"]]
        .merge(forecast[["ds", "yhat1"]], on="ds", how="left")
        .dropna()
    )

    # Prophet-style horizon (days after cutoff)
    df_eval["horizon"] = (df_eval["ds"] - cutoff).dt.days

    # Collect raw CV rows (equivalent to df_cv)
    for _, r in df_eval.iterrows():
        rows.append({
            "cutoff": cutoff,
            "horizon": r["horizon"],
            "mae": abs(r["y"] - r["yhat1"]),
            "rmse_sq": (r["y"] - r["yhat1"]) ** 2,
        })

    cutoff += period

# Raw CV table (Prophet df_cv equivalent)
df_cv_np = pd.DataFrame(rows)

# Prophet-style performance_metrics aggregation
df_perf_np = (
    df_cv_np
    .groupby("horizon")
    .agg(
        mae=("mae", "mean"),
        rmse=("rmse_sq", lambda x: (x.mean()) ** 0.5),
    )
    .reset_index()
)

df_perf_np

(To read this markdown, click on it to see the organized table better)

Note: Prophet got me 

	horizon	mse	        rmse	    mae	        mdape	    smape	    coverage
0	2 days	32.699590	5.718356	4.314039	0.216479	0.256824	0.781212
1	3 days	32.754585	5.723162	4.521047	0.235584	0.268660	0.775510
2	4 days	33.778423	5.811921	4.512585	0.236168	0.262122	0.760804
3	5 days	26.483908	5.146252	4.132443	0.218116	0.258400	0.828631
4	6 days	27.724109	5.265369	4.172599	0.202181	0.268864	0.842437
5	7 days	33.867401	5.819571	4.675453	0.208648	0.268112	0.807323
6	8 days	30.572442	5.529235	4.402987	0.195637	0.278223	0.795918
7	9 days	28.944590	5.380018	4.373379	0.226303	0.273057	0.810624
8	10 days	34.047113	5.834990	4.738925	0.294259	0.309894	0.772209
9	11 days	28.603353	5.348210	4.287819	0.218470	0.273932	0.813926
10	12 days	29.858016	5.464249	4.530661	0.249617	0.290872	0.748499
11	13 days	31.209593	5.586555	4.469629	0.248398	0.277162	0.817227
12	14 days	40.528933	6.366234	5.130353	0.239707	0.323118	0.754202

in comparison to the neural_prophet which got me 

    horizon	mae	    rmse
0	1	4.319824	5.243526
1	2	4.281804	5.173223
2	3	4.274418	5.092992
3	4	4.762128	5.819862
4	5	5.916397	7.533095
5	6	4.696809	5.932300
6	7	5.618625	7.149585
7	8	4.336979	5.405299
8	9	4.651238	6.183706
9	10	4.718530	5.936762
10	11	5.045416	6.382915
11	12	3.860458	4.520055
12	13	4.403077	5.513603
13	14	4.713475	5.816390

I do not get the impression that Neural Prophet is better enough to be worth using.